# BKinD on JHMDB (split 1) — Kaggle, two runs at once

Trains BKinD on JHMDB and scores it. With **GPU T4 x2**, two separate runs train side by side,
one per GPU (much faster than splitting one run across both GPUs). Every run uses the experiment
standard from `config/JHMDB.yaml`: batch 32, mixed precision, 30 epochs. One run takes ~4–5 h.

**Setup (once):**
1. Upload `JHMDB_rgb.zip`, `joint_positions.zip` and `splits.zip` as one **private** Kaggle dataset.
2. In this notebook: **Add Input** → your dataset.
3. **Settings:** Accelerator = **GPU T4 x2** (not P100: recent PyTorch builds may not support it); **Internet = On** (needed to download the code, packages and ImageNet weights).
4. If the GitHub repo is private: **Add-ons → Secrets**, add `GITHUB_TOKEN` (a GitHub token with read access to the repo).

**Choose the runs** in the Settings cell (`RUNS`): each has a unique `name` and a `seed`.

**Run it in the background:** *Save Version → Save & Run All (Commit)*. It keeps running after you close the browser (up to 12 h), and everything in `/kaggle/working/` is saved as this notebook's output. Training stops itself cleanly before the 12 h limit.

**Resuming a cut-off run:** add the previous version's output as an input (**Add Input → Your Work → this notebook**) and keep the same run names. Each run continues from its last saved epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ---- Settings ----
REPO_URL = 'https://github.com/sleekEagle/noisykeypoints.git'
# One run per GPU. A run's name is its output folder, so keep names unique and unchanged when resuming.
RUNS = [dict(name='JHMDB_split1_clean_s1', seed=1),
        dict(name='JHMDB_split1_clean_s2', seed=2)]
MAX_HOURS = 11.5                 # Kaggle ends a session at 12 h
DATA = '/kaggle/temp'            # scratch space, not saved
OUT = '/kaggle/working/runs'     # saved as notebook output
import os, time
T0 = time.time()
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)

In [ ]:
# Get the code (uses the GITHUB_TOKEN secret if the repo is private).
import subprocess
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    print('no GITHUB_TOKEN secret; cloning as a public repo')
CODE = f'{DATA}/noisykeypoints'
if not os.path.exists(CODE):
    subprocess.run(['git', 'clone', '-q', url, CODE], check=True)
!git -C {CODE} log -1 --format='code version: %h %s'

In [ ]:
# Kaggle already has torch, torchvision, scipy, sklearn, opencv, h5py, pyyaml, pandas, seaborn.
!pip install -q piq easydict

In [ ]:
# Find the JHMDB pieces under /kaggle/input (Kaggle usually unzips uploads; plain zips are unzipped here).
import glob, zipfile

def find_dir(test):
    for root, dirs, files in os.walk('/kaggle/input'):
        if test(root, dirs, files):
            return root

def find_or_unzip(name, test):
    d = find_dir(test)
    if d is None:
        z = glob.glob(f'/kaggle/input/**/{name}.zip', recursive=True)
        assert z, f'could not find {name} (folder or zip) under /kaggle/input'
        out = f'{DATA}/unzipped/{name}'
        zipfile.ZipFile(z[0]).extractall(out)
        d = next(r for r, ds, fs in os.walk(out) if test(r, ds, fs))
    print(f'{name}: {d}')
    return d

# frames: a folder holding the 21 action folders, e.g. brush_hair/<clip>/image_00001.jpg
frames = find_or_unzip('JHMDB_rgb', lambda r, ds, fs: 'brush_hair' in ds and
                       os.path.exists(os.path.join(r, 'brush_hair', sorted(os.listdir(os.path.join(r, 'brush_hair')))[0], 'image_00001.jpg')))
# joints: a folder holding action folders with <clip>/joint_positions.mat
joints = find_or_unzip('joint_positions', lambda r, ds, fs: 'brush_hair' in ds and
                       glob.glob(os.path.join(r, 'brush_hair', '*', 'joint_positions.mat')))
# splits: a folder with brush_hair_test_split1.txt
splits = find_or_unzip('splits', lambda r, ds, fs: 'brush_hair_test_split1.txt' in fs)

# Assemble the layout prepare_jhmdb_bkind.py expects, using symlinks (no copying).
J = f'{DATA}/JHMDB'
os.makedirs(J, exist_ok=True)
for name, src in [('frames', frames), ('joint_positions', joints), ('splits', splits)]:
    if not os.path.exists(f'{J}/{name}'):
        os.symlink(src, f'{J}/{name}')

In [ ]:
# Build BKinD's folder layout (train/ and val/ for split 1). Frames are copied (32 at a time) out of
# the slow, read-only /kaggle/input. clips.json is written last, so a folder without it is a
# half-finished build: delete it and start again.
import shutil
SPLIT = f'{DATA}/JHMDB_bkind/split1'
if os.path.exists(SPLIT) and not os.path.exists(f'{SPLIT}/clips.json'):
    print('removing an incomplete build'); shutil.rmtree(SPLIT)
if not os.path.exists(f'{SPLIT}/clips.json'):
    t = time.time()
    !python {CODE}/scripts/prepare_jhmdb_bkind.py --jhmdb {J} --out {DATA}/JHMDB_bkind --split 1
    print(f'took {(time.time() - t) / 60:.1f} min')
assert os.path.exists(f'{SPLIT}/clips.json'), 'data preparation failed (see the output above)'
print('data ready:', len(os.listdir(f'{SPLIT}/train')), 'train clips,', len(os.listdir(f'{SPLIT}/val')), 'val clips')

In [ ]:
# One config per run. Paths, seed and time limit change; the experiment settings (batch 32,
# mixed precision, epochs, ...) come unchanged from config/JHMDB.yaml.
import yaml, torch
BK = f'{CODE}/third_party/BKinD'
base = yaml.safe_load(open(f'{BK}/config/JHMDB.yaml'))
n_gpu = torch.cuda.device_count()
assert len(RUNS) <= n_gpu, f'{len(RUNS)} runs but only {n_gpu} GPU(s): one run per GPU'
# Leave ~40 min of the session for setup already spent, scoring and saving the output.
train_hours = MAX_HOURS - (time.time() - T0) / 3600 - 0.7
for i, r in enumerate(RUNS):
    r['dir'] = f'{OUT}/{r["name"]}'
    os.makedirs(r['dir'], exist_ok=True)
    cfg = dict(base, data=SPLIT, checkpoint=r['dir'], gpu=0, workers=2,   # 2 loader workers each: Kaggle has 4 CPUs
               seed=r['seed'], max_hours=round(train_hours, 2))
    r['config'] = f'{BK}/config/kaggle_{r["name"]}.yaml'
    yaml.safe_dump(cfg, open(r['config'], 'w'))
    print(f'GPU {i}: {r["name"]}  seed={r["seed"]}  batch={cfg["batch_size"]}  amp={cfg["amp"]}  '
          f'epochs={cfg["epochs"]}  time limit={cfg["max_hours"]} h')

In [ ]:
# Train all runs at once, one per GPU. If a previous version's output is attached as input,
# each run resumes from its own checkpoint. Progress is printed every 2 minutes.
for i, r in enumerate(RUNS):
    ckpt = f'{r["dir"]}/checkpoint.pth.tar'
    if not os.path.exists(ckpt):
        prev = glob.glob(f'/kaggle/input/**/runs/{r["name"]}/checkpoint.pth.tar', recursive=True)
        if prev:
            shutil.copytree(os.path.dirname(prev[0]), r['dir'], dirs_exist_ok=True)  # keeps log.txt, samples
            print(f'{r["name"]}: copied previous run from {os.path.dirname(prev[0])}')
    cmd = ['python', 'train_video.py', '--config', r['config']]
    if os.path.exists(ckpt):
        cmd += ['--resume', ckpt]
    env = dict(os.environ, PYTHONUNBUFFERED='1', CUDA_VISIBLE_DEVICES=str(i))  # this run sees only GPU i
    r['log'] = open(f'{r["dir"]}/train_log.txt', 'a')
    r['proc'] = subprocess.Popen(cmd, cwd=BK, env=env, stdout=r['log'], stderr=subprocess.STDOUT)
    print(f'GPU {i}: started {r["name"]}', '(resuming)' if '--resume' in cmd else '(fresh)')

def last_line(path, keys=('Epoch:', 'epoch ', 'STOPPED', 'COMPLETE', 'Error')):
    lines = [l.rstrip() for l in open(path, errors='replace') if l.startswith(keys) or 'Error' in l]
    return lines[-1] if lines else '(starting)'

while any(r['proc'].poll() is None for r in RUNS):
    time.sleep(120)
    print(time.strftime('%H:%M'), f'({(time.time() - T0) / 3600:.1f} h into session)')
    for r in RUNS:
        print(f'   {r["name"]}: {last_line(r["dir"] + "/train_log.txt")}')
for r in RUNS:
    r['log'].close()
    print(f'{r["name"]}: exit code {r["proc"].returncode}')
    if r['proc'].returncode != 0:
        !tail -n 30 {r['dir']}/train_log.txt

In [ ]:
# Training curve per run: learning rate, train loss, val loss per epoch.
for r in RUNS:
    print('=====', r['name']); print(open(f'{r["dir"]}/log.txt').read())

In [ ]:
# Score each finished run on the val (official test) clips. A run stopped by the time limit is
# skipped: resume it in a new session first.
%cd {CODE}
for r in RUNS:
    ck = torch.load(f'{r["dir"]}/checkpoint.pth.tar', map_location='cpu', weights_only=False)
    if ck['epoch'] < base['epochs']:
        print(f'{r["name"]}: only {ck["epoch"]}/{base["epochs"]} epochs done; resume before scoring'); continue
    print('=====', r['name'])
    !python scripts/eval_bkind_jhmdb.py --checkpoint {r['dir']}/checkpoint.pth.tar --data {SPLIT} --jhmdb {J} --out {r['dir']}/eval

In [ ]:
from IPython.display import Image, display
for r in RUNS:
    p = f'{r["dir"]}/eval/vis_val.png'
    if os.path.exists(p):
        print(r['name']); display(Image(p))